# [LEC] Streamlit을 활용한 RAG 배포

슬라이드와 설명, 퀴즈로 RAG 체인을 웹 화면에 붙이는 Streamlit의 기본 구조를 익힙니다.

**목차**

- 1. Streamlit (슬라이드) (퀴즈)
  - 1.1 Streamlit 소개 (슬라이드) (퀴즈)
  - 1.2 Python 코드 기반 웹 UI (슬라이드) (퀴즈)
- 2. Streamlit 앱 기본 구성 (슬라이드) (퀴즈)
  - 2.1 title과 caption (슬라이드)
  - 2.2 cache_resource (슬라이드) (퀴즈)
- 3. 채팅 인터페이스 구성 (슬라이드) (퀴즈)
  - 3.1 session_state (슬라이드) (퀴즈)
  - 3.2 chat_message와 chat_input (슬라이드) (퀴즈)
  - 3.3 write_stream (슬라이드) (퀴즈)
- 4. 정리 퀴즈 (퀴즈)

In [ ]:
import sys

sys.path.insert(0, "..")  # 챕터 폴더의 assets 도구를 불러오기 위한 경로

from assets.widgets import show_quiz, show_slides, slide_setup

slide_setup()  # 슬라이드 뷰어(PDF.js)를 불러옵니다. 슬라이드가 비어 보이면 이 셀을 다시 실행합니다.

In [ ]:
show_slides("2-3", [1, 2])

이번 시간은 앞에서 만든 RAG 체인을 웹 화면에 붙이는 방법을 세 부분으로 봅니다.

1. Streamlit: 무엇이고, 코드가 어떻게 실행되며 화면을 그리는가
2. Streamlit 앱 기본 구성: 제목·안내 문구(title·caption)와 무거운 준비를 한 번만 하는 cache_resource
3. 채팅 인터페이스 구성: 대화 기록(session_state), 말풍선과 입력창(chat_message·chat_input), 답변 스트리밍(write_stream)

예시 앱은 보험 약관을 근거로 답하는 "보험 약관 QA 챗봇"입니다.

## 1. Streamlit <small>(슬라이드) (퀴즈)</small>

### 1.1 Streamlit 소개 <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-3", [3, 4])

Streamlit은 Python 코드만으로 웹 애플리케이션을 빠르게 만드는 프레임워크입니다.
HTML, CSS, JavaScript를 직접 쓰지 않고, Python 함수와 변수를 화면 요소로 바로 연결합니다.
데이터 분석, AI 데모, 내부 업무 도구를 빠르게 만들 때 알맞습니다.

LLM 실습에서 Streamlit을 쓰는 이유는 네 가지입니다.
- 콘솔 실행 결과를 사용자가 보기 쉬운 화면으로 바꿉니다.
- 질문 입력창과 답변 영역을 간단히 만듭니다.
- 모델 응답이 생성되는 과정을 스트리밍으로 보여 줍니다.
- 실습한 RAG 체인을 웹 화면과 연결해 서비스 형태로 확인합니다.

**왜 배우나:** 지금까지 RAG 체인의 답은 콘솔에 출력했습니다. 다른 사람이 쓰려면 질문을 넣고 답을 읽는 화면이 필요합니다. Streamlit은 그 화면을 Python만으로 붙이는 마지막 단계입니다.

In [ ]:
show_quiz("../assets/quizzes/2-3/q11.json")

### 1.2 Python 코드 기반 웹 UI <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-3", [5])

Streamlit 앱은 위에서 아래로 실행되는 Python 스크립트처럼 씁니다.
터미널에서 `streamlit run` 명령에 앱 파일 이름을 붙여 실행하면 코드가 실행되며 화면 요소가 순서대로 그려집니다.

일반 웹 앱과 다르게 생각해야 할 점이 있습니다. 버튼을 누르거나 입력을 하면 **스크립트 전체가 처음부터 다시 실행**됩니다.
그래서 유지해야 하는 값은 `st.session_state`에 저장하고, 오래 걸리는 준비 작업은 cache 기능으로 재사용합니다.

**왜 배우나:** 이 "전체 재실행" 구조가 뒤에 나오는 cache_resource와 session_state가 필요한 이유입니다. 두 기능은 모두 재실행 때 무엇을 남길지 정하는 도구입니다.

In [ ]:
show_quiz("../assets/quizzes/2-3/q12.json")

## 2. Streamlit 앱 기본 구성 <small>(슬라이드) (퀴즈)</small>

### 2.1 title과 caption <small>(슬라이드)</small>

In [ ]:
show_slides("2-3", [6, 7, 8])

`st.title`은 앱의 가장 큰 제목을 화면에 출력합니다. 사용자가 어떤 앱을 보고 있는지 바로 알게 하는, 화면의 첫인상을 정하는 요소입니다. 예시 앱은 "보험 약관 QA 챗봇"을 제목으로 씁니다.

`st.caption`은 제목 아래에 짧은 보조 설명을 출력합니다. 앱이 어떤 데이터를 근거로 답하는지 안내하는 데 씁니다. 예시 앱은 "메리츠 펫보험 약관을 기반으로 답변합니다."를 출력합니다. 제목보다 작고 가벼운 설명에 알맞습니다.

### 2.2 cache_resource <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-3", [9])

`@st.cache_resource`는 무거운 리소스를 한 번만 만들고 재사용하게 하는 데코레이터(함수 위에 붙여 동작을 더하는 표시)입니다.
모델, 벡터스토어, DB 연결처럼 만드는 데 오래 걸리는 객체에 씁니다.
Streamlit이 다시 실행되어도 같은 리소스를 다시 만들지 않도록 결과를 캐시에 보관합니다.
예시 앱은 PDF를 읽고 벡터스토어와 RAG 체인을 만드는 `load_rag_chain()`에 붙입니다.

**왜 배우나:** 이 표시가 없으면 질문할 때마다 스크립트가 다시 실행되면서 PDF를 다시 읽고 임베딩을 다시 만듭니다. 답이 느려지고 임베딩 호출 비용도 매번 듭니다.

In [ ]:
show_quiz("../assets/quizzes/2-3/q22.json")

## 3. 채팅 인터페이스 구성 <small>(슬라이드) (퀴즈)</small>

### 3.1 session_state <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-3", [10, 11])

`st.session_state`는 사용자 세션 동안 유지해야 하는 값을 저장하는 공간입니다.
Streamlit 앱은 입력이 생길 때 다시 실행되므로, 일반 변수에만 담으면 이전 대화가 사라집니다.
대화 기록처럼 유지해야 하는 값은 `session_state`에 저장합니다.
그리고 화면은 재실행 때마다 새로 그려지므로, 저장된 메시지를 매번 처음부터 다시 출력합니다. 슬라이드 코드의 `for msg in st.session_state.messages:` 부분이 그 일을 합니다.

**왜 배우나:** 이 대화 기록은 **화면에 보여 주기 위한 것**입니다. 예시 앱은 체인에 이번 질문만 넣으므로, LLM이 이 기록을 보고 이전 대화를 기억하지는 않습니다. "그건 얼마예요?"처럼 앞 대화를 가리키는 질문을 하면 LLM은 "그것"이 무엇인지 모릅니다. 대화를 기억하게 하려면 이전 대화를 요청에 다시 넣어 보내야 합니다.

In [ ]:
show_quiz("../assets/quizzes/2-3/q31.json")

### 3.2 chat_message와 chat_input <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-3", [12])

`st.chat_message`는 채팅 메시지를 역할별 말풍선으로 출력합니다. `user` 역할은 사용자 메시지 영역, `assistant` 역할은 AI 답변 영역입니다.

`st.chat_input`은 화면 하단에 질문 입력창을 만듭니다. 사용자가 입력하고 Enter를 누르면 입력한 문자열을 돌려줍니다.
슬라이드 코드는 `if prompt := st.chat_input(...)`로 입력이 있을 때만 아래를 실행합니다. 받은 질문을 먼저 `session_state`에 저장하고, 그다음 user 말풍선으로 출력합니다.

In [ ]:
show_quiz("../assets/quizzes/2-3/q32.json")

### 3.3 write_stream <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("2-3", [13])

`st.write_stream`은 만들어지는 텍스트를 한 번에 보여 주지 않고 차례로 화면에 출력합니다.
LLM이 답변을 만드는 과정을 실시간으로 보여 주므로, 사용자는 응답이 시작되었다는 것을 바로 알 수 있습니다. 긴 답변일수록 체감 대기 시간이 줄어듭니다.

슬라이드 코드는 assistant 말풍선 안에서 `load_rag_chain()`으로 체인을 가져오고, `rag_chain.stream(prompt)`의 조각을 `st.write_stream`에 넘깁니다. 출력이 끝나면 전체 답변이 `response`에 담기고, 이것을 assistant 메시지로 `session_state`에 저장합니다.

**왜 배우나:** LLM 체인의 `stream()`은 답변을 조각으로 돌려줍니다. `st.write_stream`은 그 조각을 화면에 바로 잇는 연결 고리입니다. 그리고 다 받은 답변을 저장해 두어야 다음 재실행 때 이 답변도 다시 그릴 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/2-3/q33.json")

## 4. 정리 퀴즈 <small>(퀴즈)</small>

In [ ]:
show_quiz("../assets/quizzes/2-3/final.json")

**이 노트북에서 본 것**
- Streamlit은 Python 코드만으로 웹 화면을 만듭니다. RAG 체인을 질문 입력창과 답변 화면에 붙이는 데 씁니다.
- 버튼을 누르거나 입력을 하면 스크립트 전체가 처음부터 다시 실행됩니다.
- `st.title`은 가장 큰 제목, `st.caption`은 제목 아래의 짧은 안내입니다.
- `@st.cache_resource`는 RAG 체인처럼 무거운 객체를 한 번만 만들어 재사용합니다.
- `st.session_state`는 대화 기록처럼 재실행 뒤에도 남길 값을 저장합니다. 이 기록은 화면 표시용이라 LLM이 기억하게 해 주지는 않습니다.
- `st.chat_message`는 역할별 말풍선, `st.chat_input`은 하단 입력창, `st.write_stream`은 답변을 차례로 출력하고 전체 답변을 돌려줍니다.